# AERIS - diagnostic augmentation Keras (tf_keras)

Verifie si le bloc d'augmentation du notebook TB/PN corrompt les images
normalisees dans [0, 1] sous **tf_keras 2.20** (l'environnement reel des
entrainements), et si `value_range=(0.0, 1.0)` corrige le tir.


In [ ]:
print('=== diagnostic : augmentation Keras dans tf_keras ===', flush=True)
import inspect
import numpy as np
import tensorflow as tf
import tf_keras
keras = tf_keras
print('tensorflow', tf.__version__)
print('keras', keras.__version__)

for nom in ['RandomBrightness', 'RandomContrast', 'RandomZoom', 'RandomRotation', 'RandomFlip']:
    couche = getattr(keras.layers, nom)
    print(nom, inspect.signature(couche.__init__), flush=True)


In [ ]:
print('=== image reelle normalisee [0,1] ===', flush=True)
from pathlib import Path

RACINE = None
for base in (Path('/kaggle/input'), Path('/kaggle/input/datasets'), Path('/kaggle/working')):
    if base.exists():
        for candidat in sorted(base.glob('**/*.png')):
            RACINE = candidat
            break
    if RACINE:
        break
print('image test', RACINE, flush=True)

IMG = keras.utils.load_img(RACINE, target_size=(96, 96), color_mode='rgb')
X = keras.utils.img_to_array(IMG) / 255.0
B = np.repeat(X[None, ...], 8, axis=0)
print('entree  min %.4f max %.4f mean %.4f std %.4f' % (B.min(), B.max(), B.mean(), B.std()), flush=True)


In [ ]:
print('=== chaque couche, appliquee a une image [0,1] ===', flush=True)


def stats(nom, sortie):
    s = np.asarray(sortie)
    print('%-45s min %8.4f max %8.4f mean %8.4f std %6.4f' % (nom, s.min(), s.max(), s.mean(), s.std()), flush=True)


stats('RandomBrightness(0.10)  [defaut]', keras.layers.RandomBrightness(0.10)(B, training=True))
stats('RandomBrightness(0.10)  value_range=(0,1)', keras.layers.RandomBrightness(0.10, value_range=(0.0, 1.0))(B, training=True))
stats('RandomContrast(0.10)    [defaut]', keras.layers.RandomContrast(0.10)(B, training=True))
stats('RandomZoom(0.10)        [defaut]', keras.layers.RandomZoom(0.10)(B, training=True))
stats('RandomRotation(10/360)  [defaut]', keras.layers.RandomRotation(10.0 / 360.0)(B, training=True))
stats('RandomFlip(horizontal)  [defaut]', keras.layers.RandomFlip('horizontal')(B, training=True))
stats('aucune couche (temoin)', B)


In [ ]:
print('=== bloc complet TB, tel quel puis corrige ===', flush=True)
bloc_telquel = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomFlip('vertical'),
    keras.layers.RandomRotation(10.0 / 360.0),
    keras.layers.RandomZoom(0.10),
    keras.layers.RandomBrightness(0.10),
    keras.layers.RandomContrast(0.10),
], name='telquel')

bloc_corrige = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomFlip('vertical'),
    keras.layers.RandomRotation(10.0 / 360.0),
    keras.layers.RandomZoom(0.10),
    keras.layers.RandomBrightness(0.10, value_range=(0.0, 1.0)),
    keras.layers.RandomContrast(0.10),
], name='corrige')

for i in range(3):
    stats('bloc tel quel tirage %d' % i, bloc_telquel(B, training=True))
for i in range(3):
    stats('bloc corrige  tirage %d' % i, bloc_corrige(B, training=True))


In [ ]:
print('=== pipeline tf.data reel (from_generator -> shuffle -> map) ===', flush=True)
AUTOTUNE = tf.data.AUTOTUNE
signature = (
    tf.TensorSpec(shape=(96, 96, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32),
)


def generateur():
    for _ in range(16):
        yield X, 0.0


ds = (
    tf.data.Dataset.from_generator(generateur, output_signature=signature)
    .shuffle(16, seed=42, reshuffle_each_iteration=True)
    .map(lambda x, y: (bloc_corrige(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .batch(8)
)
lot_x, lot_y = next(iter(ds))
stats('lot pipeline corrige', lot_x)
print('valeurs hors [0,1] :', int((np.asarray(lot_x) < 0).sum() + (np.asarray(lot_x) > 1).sum()), flush=True)
print('diagnostic termine', flush=True)
